# MicroDuck × Qwen-VL · 实时仿真
现场输入中文，观察 Qwen3-VL-4B + `lang-v2` 的逐步决策、MuJoCo 实时第一视角和返回起点的距离。

在 **运行时 → 更改运行时类型** 选择 **GPU（优先 L4）**（支持 Python 3.12 和 3.13）。按顺序运行第 1–3 格预热，第 4 格留到现场。第一次运行需下载基座模型。每次运行第 4 格都会重置地图，模型保留在显存中。

默认从公开 GitHub 仓库取源码，无需挂载 Drive。也可用第一格的 `project_source` 选择 Drive 或上传 ZIP。首次安装后如出现包导入错误，重启会话后再运行所有单元，已下载文件会保留。适配器与步态从公开的 Hugging Face 仓库拉取，免登录。

In [ ]:
# @title 1 · 取出项目源码并安装依赖
project_source = "github"  # @param ["github", "drive", "upload"]
drive_folder = "microduck-qwen"  # @param {type:"string"}
render_backend = "egl"  # @param ["egl", "osmesa"]

from importlib import metadata
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
import zipfile
from pathlib import Path

if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("请选择 Python 3.12 或 3.13 的 GPU 运行时，然后重新运行。")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("请启用 Colab GPU（优先 L4），然后重新运行。")
print("GPU:", gpu.stdout.strip())
root = Path("/content/microduck-qwen-github" if project_source == "github" else "/content/microduck-qwen")
bundle = "microduck-qwen-colab-source.zip"
local_zip = Path("/content") / bundle
if project_source == "github":
    if not (root / "pyproject.toml").is_file():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/atlasmarkz/microduck-qwen.git", str(root)], check=True)
else:
    if local_zip.is_file():
        print("使用已有的源码包。")
    elif project_source == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        source = Path("/content/drive/MyDrive") / drive_folder / bundle
        if not source.is_file():
            raise FileNotFoundError(f"未找到 {source}，也可切换到 github 或 upload。")
        shutil.copyfile(source, local_zip)
    else:
        from google.colab import files
        uploaded = files.upload()
        if bundle not in uploaded:
            raise FileNotFoundError(f"请选择 {bundle}。")
        local_zip.write_bytes(uploaded[bundle])
    with zipfile.ZipFile(local_zip) as archive:
        archive.extractall(root)

os.environ["MUJOCO_GL"] = render_backend
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir(root)
sys.path.insert(0, str(root))

def run(*cmd):
    completed = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if completed.returncode:
        raise RuntimeError(completed.stdout[-8500:])
    if completed.stdout.strip():
        print(completed.stdout[-2500:], flush=True)

gl_packages = ("libegl1", "libgl1") if render_backend == "egl" else ("libosmesa6", "libgl1")
subprocess.run(["apt-get", "-qq", "update"], check=True)
run("apt-get", "-qq", "-y", "install", *gl_packages)
project = tomllib.loads((root / "pyproject.toml").read_text())["project"]
requirements = project["optional-dependencies"]
bam = Path("/content/microduck-bam.zip")
bam_url = next(item.split(" @ ", 1)[1] for item in requirements["simulation"] if " @ " in item)
run("curl", "-fsSL", "--retry", "3", bam_url, "-o", str(bam))
sim_packages = [item for item in requirements["simulation"] if " @ " not in item]
print("预装 PyTorch:", metadata.version("torch"), flush=True)
with tempfile.TemporaryDirectory() as build:
    bam_source = bam
    if sys.version_info[:2] == (3, 13):
        # This demo uses only BAM's NumPy and MuJoCo path. Keep its pinned code
        # and model parameters, and widen just the source build's Python metadata.
        with zipfile.ZipFile(bam) as archive:
            archive.extractall(build)
        manifest, = Path(build).glob("bam-*/pyproject.toml")
        config = manifest.read_text(encoding="utf-8")
        guard = 'requires-python = ">=3.12,<3.13"'
        if config.count(guard) != 1:
            raise RuntimeError("固定版 BAM 的 Python 声明与预期不符。")
        manifest.write_text(config.replace(guard, 'requires-python = ">=3.12,<3.14"'),
                            encoding="utf-8")
        bam_source = manifest.parent
    run(sys.executable, "-m", "pip", "install", "-q", "Pillow>=10.1", *project["dependencies"],
        *[p for p in requirements["vision-torch"] if not p.startswith("Pillow==")],
        *sim_packages, str(bam_source))
print("依赖安装完成。")

In [ ]:
# @title 2 · 下载固定版本的模型和仿真资产
import tarfile
import tempfile
from huggingface_hub import hf_hub_download, snapshot_download
from microduck_qwen.sim_native import DEFAULT_POLICY, DEFAULT_RL, POLICY_REVISION
from microduck_qwen.vision_torch import MODEL, MODEL_PATH, REVISION, checked_adapter

adapter_repo = "atlasmarkz/microduck-skill-agent"
adapter_rev = "f5924d1576b6e5b852320994b0e1db911becd5e6"
assets_dir = root / ".cache" / "colab-adapters"
snapshot_download(adapter_repo, revision=adapter_rev, local_dir=assets_dir,
                  allow_patterns=["lang-v2/*"], token=False)
adapter = assets_dir / "lang-v2"
adapter_report = checked_adapter(adapter, "agent")
print("lang-v2 SHA-256:", adapter_report["adapter_sha256"])

hf_hub_download("pollen-robotics/microduck-policies", "alpha_walking.onnx",
                revision=POLICY_REVISION, cache_dir=DEFAULT_POLICY.parents[3], token=False)

upstream_rev = "0daab160d3fb1c2cbaba16a4a19052f36709dd9e"
if not DEFAULT_RL.exists():
    DEFAULT_RL.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=DEFAULT_RL.parent) as temporary:
        upstream_archive = Path(temporary) / "upstream.tar.gz"
        run("curl", "-fsSL", "--retry", "3",
            f"https://codeload.github.com/pollen-robotics/microduck_rl/tar.gz/{upstream_rev}",
            "-o", str(upstream_archive))
        with tarfile.open(upstream_archive) as archive:
            archive.extractall(temporary, filter="data")
        (Path(temporary) / f"microduck_rl-{upstream_rev}").rename(DEFAULT_RL)

snapshot_download(MODEL, revision=REVISION, cache_dir=MODEL_PATH.parents[2],
                  allow_patterns=["*.json", "*.safetensors", "merges.txt", "vocab.json", "chat_template.jinja"],
                  token=False)
assert DEFAULT_POLICY.is_file() and MODEL_PATH.is_dir(), "仿真策略或基座模型下载不完整。"
print("模型和仿真资产就绪。")

In [ ]:
# @title 3 · 预热模型，检查真实 MuJoCo 图像和推理
from IPython.display import display
from PIL import Image
from tempfile import TemporaryDirectory

from microduck_qwen.agent import memory_state
from microduck_qwen.colab_demo import DEMO_INSTRUCTION, DEMO_SEED, simulation
from microduck_qwen.navigation import Observer
from microduck_qwen.vision import capture, propose_targets, save_frame
from microduck_qwen.vision_torch import TorchAgentPlanner

if "planner" not in globals():
    planner = TorchAgentPlanner(adapter)
with simulation(DEMO_SEED) as info, TemporaryDirectory() as scratch:
    observer = Observer(info["port"])
    try:
        state = observer.read()
    finally:
        observer.close()
    frame = capture(info["port"])
    image_path = save_frame(Path(scratch), frame, propose_targets(frame), "warmup")
    preview = planner.predict(DEMO_INSTRUCTION, image_path, memory_state(info, state, []))
    if preview.get("action") is None:
        raise RuntimeError(f"模型预热未生成可执行动作：{preview.get('error', preview.get('raw'))}")
    display(Image.open(image_path).copy())
    print("预热完成，首个动作：", preview["action"])

In [ ]:
# @title 4 · 会议现场：修改指令后运行
instruction = "去别的房间找蓝色的球，走到它旁边，然后回起点"  # @param {type:"string"}
seed = 98107  # @param {type:"integer"}

from microduck_qwen.colab_demo import live_demo
report = live_demo(planner, instruction, seed)
print("完整决策和执行记录：", Path(report["directory"]) / "result.json")

完整演示约两分钟，受 GPU、渲染性能和任务路径影响。如果出现 EGL 初始化错误，第 1 格可改为 `osmesa`，重新启动运行时并重跑。第 4 格停止执行会停止仿真。地图和轨迹使用仿真位姿，仅用于展示。模型看到的是机器人相机图像；`done` 表示模型声明任务完成。